# Сводка результатов файнтюна

Проходит по всем `results/finetune/*.json`, складывает в один CSV (`results/all_methods_comparison.csv`) и рисует top-N бары по метрикам.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'results' / 'finetune'
OUT_CSV = PROJECT_ROOT / 'results' / 'all_methods_comparison.csv'

TOP_N = 10
ROUND = 4

print(f'Сканирую: {RESULTS_DIR}')

In [ ]:
rows = []
for path in sorted(RESULTS_DIR.glob('*.json')):
    with open(path, 'r', encoding='utf-8') as f:
        r = json.load(f)
    rows.append({
        'run_key': r.get('run_key', path.stem),
        'method': r.get('method'),
        'model': r.get('model'),
        'balanced_accuracy': r.get('balanced_accuracy'),
        'macro_f1': r.get('macro_f1'),
        'f1_group_A': r.get('f1_group_A'),
        'f1_group_B': r.get('f1_group_B'),
        'f1_group_C': r.get('f1_group_C'),
        'trainable_params': r.get('trainable_params'),
        'train_time_sec': r.get('train_time_sec'),
        'timestamp': r.get('timestamp'),
    })

df = pd.DataFrame(rows)

# Округляем все числовые метрики до ROUND знаков
metric_cols = ['balanced_accuracy', 'macro_f1', 'f1_group_A', 'f1_group_B', 'f1_group_C']
for c in metric_cols:
    df[c] = df[c].astype(float).round(ROUND)

df = df.sort_values('macro_f1', ascending=False).reset_index(drop=True)

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print(f'Сохранено: {OUT_CSV} ({len(df)} прогонов)')
df

In [ ]:
def plot_top(df, metric, title, color):
    top = df.sort_values(metric, ascending=False).head(TOP_N).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(top))))
    ax.barh(top['run_key'], top[metric], color=color)
    ax.set_xlabel(metric)
    ax.set_title(f'Top-{TOP_N} | {title}')
    for i, v in enumerate(top[metric]):
        ax.text(v, i, f' {v:.4f}', va='center', fontsize=9)
    ax.set_xlim(0, max(top[metric].max() * 1.15, 0.05))
    plt.tight_layout()
    plt.show()

if not df.empty:
    plot_top(df, 'balanced_accuracy', 'Balanced Accuracy', 'steelblue')
    plot_top(df, 'macro_f1',          'Macro F1',          'salmon')
else:
    print('Нет результатов для построения графиков.')